# Statistical significance testing for snow depth distributions


In [ ]:
# Conduct statistical tests to determine whether snow depth distributions are significantly different
# Read in snow depth datasets for each of the ASO flights
# For each basin and flight, you should have ASO, Baseline iSnobal, HRRR-SPIReS, NWM (maybe), and UA

In [ ]:
import sys
import scipy
import numpy as np

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# First set up the directory with the depth data
in_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ASO/diffs'

# Set the snow property variable and the projection type
snowprop = 'depth'
reproj_type = 'uniformreproj'

In [ ]:
# Locate difference files
diff_fn_list = h.fn_list(in_dir, f'*_wy*_{snowprop}_{reproj_type}.nc')
# Get a unique list of the dates in the filenames
diff_dates = sorted(list(set([f.split('_')[4] for f in diff_fn_list])))
# diff_dates
big_dt_depth_list = []
# Now go through each dt and put together a dt list
for dt in diff_dates:
    dt_depth_list = []
    dt_fn_list = h.fn_list(in_dir, f'*{dt}*{snowprop}_{reproj_type}.nc')
    # Remove the prtest bit if it is in there
    dt_fn_list = [f for f in dt_fn_list if 'prtest' not in f]
    # Read in the datasets
    dt_ds_list = [h.load(f) for f in dt_fn_list]
    # Combine in pairs to re-create the depth datasets
    # Create the depth arrays by adding the array values together in pairs

    for i in range(0, len(dt_ds_list), 2):
        ds1 = dt_ds_list[i]
        ds2 = dt_ds_list[i+1]
        new_depth = ds1 + ds2
        new_depth.attrs['description'] = f'{ds2.description.split(' - ')[0]} snow depth [m]'
        new_depth.attrs['basin'] = ds1.basin
        new_depth.attrs['WY'] = ds2.WY
        new_depth.attrs['dt'] = dt
        new_depth.attrs['date'] = ds2.date
        new_depth.attrs['model'] = f'recombined {ds2.description.split(' - ')[0]}'
        new_depth.name = f'{ds2.description.split(' - ')[0]}'
        dt_depth_list.append(ds1) # add ASO depth
        dt_depth_list.append(new_depth) # add reconstituted depth
    big_dt_depth_list.append(dt_depth_list)

In [ ]:
# Get some random integers
# sample_size = 500000
# sample_size = 100000
# sample_size = 10000
# sample_size = 1000
rng = np.random.default_rng()
# rand_idx = rng.integers(low=0, high=ds1.size, size=sample_size)
rand_idx = rng.integers(low=0, high=ds1.size, size=int(ds1.size/10))
# Do some clean up
del ds1, ds2, new_depth, dt_depth_list

In [ ]:
pval = 0.05
# For each list of depths per date, let's do some comparison testing
big_mwu_results = []
for dt, dt_depth_list in zip(diff_dates, big_dt_depth_list):
    basin = dt_depth_list[0].basin
    print('\n', basin, dt)
    # Loop through the paired datasets
    mwu_results = []
    for i in range(0, len(dt_depth_list), 2):
        ds1 = dt_depth_list[i].load()
        ds2 = dt_depth_list[i+1].load()
        # Test if the means are different using a Mann-Whitney U test
        # Null: the means are the same
        x = ds1.values.flatten()
        y = ds2.values.flatten()
        # Randomly select samples from the distributions using the same indices
        rand_x = x[rand_idx]
        rand_y = y[rand_idx]
        res = scipy.stats.mannwhitneyu(x=rand_x, y=rand_y, nan_policy='omit')
        if res.pvalue < pval:
            sigprint = "-- Reject H0, means are significantly different !!"
        else:
            sigprint = "Cannot reject H0"
        print(f'{ds2.name}: {res.statistic}, {res.pvalue:.2e} {sigprint}')
        mwu_results.append([basin, dt, ds2.name, res.statistic, res.pvalue])
    big_mwu_results.append(mwu_results)

In [ ]:
# TODO: if significantly different, determine if the distribution is less than or greater than the other

In [ ]:
results_df = pd.DataFrame(h.flatten(big_mwu_results), columns=['basin', 'dt', 'model', 'statistic', 'pvalue'])
# Rearrange by basin and date
results_df = results_df.sort_values(by=['basin', 'dt'])

# Add a column for significance
results_df['significant'] = results_df['pvalue'] < pval
results_df

In [ ]:
# For each basin
# Plot the pvalues for each model and date, with a horizontal line at the p-value threshold for significance
# Instead of using hue to signify models, use bar fill: no fill, hatched fill, dotted fill, and complete fill
bar_dict = {'Baseline' : ['...', 'lavender'],
              'HRRR-SPIReS' : ['', 'darkblue'],
              'NWM' : ['', 'mediumpurple'],
              'UA' : ['', 'xkcd:golden']}
for basin in results_df['basin'].unique():
    basin_df = results_df[results_df['basin'] == basin]
    fig, ax = plt.subplots(figsize=(max(4, len(basin_df['dt'].unique())), 3))
    for i, model in enumerate(basin_df['model'].unique()):
        data = basin_df[basin_df['model']==model].set_index('dt')['significant']
        ax.bar(x=np.arange(len(data)) + i*0.2, height=data.values, width=0.2, label=model,
               hatch=bar_dict[model][0], edgecolor='black', color=bar_dict[model][1])
    dates = sorted(basin_df['dt'].unique())
    ax.set(xlabel='Date', ylabel='Significant', title=f'{basin}',
           xticks=np.arange(len(dates))+0.3, xticklabels=dates)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize=8)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
#     plt.savefig(f'/uufs/chpc.utah.edu/common/home/u6058223/public_html/manuscript_figs/{basin}_{dt}_significance.png', dpi=300)